# EX_06 — Introducción a RAG (ejercicios)

**Notebook de referencia:** `notebook/06_Introduccion_RAG.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Plantilla de contexto

Escribe una función `build_prompt(context_chunks, question) -> str` que inserte los pasajes en un delimitador claro (`### Context` / `### Question`).


In [1]:
def build_prompt(context_chunks: list[str], question: str) -> str:
    # 1. Unimos todos los fragmentos de la lista en un solo bloque de texto, 
    # separados por saltos de línea para mayor legibilidad.
    contexto_unido = "\n".join(context_chunks)
    
    # 2. Construimos la plantilla final usando los delimitadores solicitados
    prompt = f"""### Context
{contexto_unido}

### Question
{question}

Respuesta:"""
    
    return prompt

# --- PRUEBA DEL CÓDIGO ---
pasajes_recuperados = [
    "- FAISS es una biblioteca para búsqueda de similitud rápida.",
    "- La similitud del coseno mide el ángulo entre vectores."
]
pregunta_usuario = "¿Para qué sirve FAISS?"

prompt_final = build_prompt(pasajes_recuperados, pregunta_usuario)
print(prompt_final)


### Context
- FAISS es una biblioteca para búsqueda de similitud rápida.
- La similitud del coseno mide el ángulo entre vectores.

### Question
¿Para qué sirve FAISS?

Respuesta:


## Actividad 2 — RAG sin LLM (retrieval only)

Con tus chunks del notebook teórico (o texto inventado), recupera top-k y **imprime** el contexto ensamblado sin llamar al generador.


In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

# 1. Preparación del "Motor" y la Base de Datos (Chunks inventados)
modelo = SentenceTransformer('all-MiniLM-L6-v2')

chunks_teoria = [
    "La mitocondria es conocida como la central energética de la célula.",
    "Python fue creado por Guido van Rossum y lanzado por primera vez en 1991.",
    "El núcleo celular contiene la mayor parte del material genético de la célula.",
    "Java es un lenguaje de programación orientado a objetos, de propósito general.",
    "El ciclo de Krebs es una ruta metabólica clave en la respiración celular."
]

# Creamos el índice FAISS y añadimos nuestros chunks
embeddings_bd = modelo.encode(chunks_teoria)
faiss.normalize_L2(embeddings_bd)
indice = faiss.IndexFlatIP(embeddings_bd.shape[1])
indice.add(embeddings_bd)

# ====================================================================
# EMPIEZA LA TAREA: Recuperar top-k e imprimir el contexto ensamblado
# ====================================================================

pregunta = "¿Qué funciones tienen los orgánulos dentro de la célula?"
top_k = 2

# A) Búsqueda Semántica (Retrieval)
vector_pregunta = modelo.encode([pregunta])
faiss.normalize_L2(vector_pregunta)

# Obtenemos los índices de los 2 chunks más relevantes
distancias, posiciones = indice.search(vector_pregunta, top_k)

# B) Ensamblaje del Contexto
# Extraemos los textos reales usando las posiciones que nos

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Actividad 3 — Fallo de cobertura

Inventa un caso donde la respuesta **no** está en los chunks recuperados y describe en español (markdown) cómo lo detectarías en producción (p. ej. umbral de score, abstención).


### Caso Práctico: El Incidente de la Nave Estelar 

**El Escenario:**
Imagina que hemos construido un sistema RAG para interactuar con el manual de emergencias de una nave espacial. 
* **Query del usuario (Astronauta en pánico):** *"¿Cómo apago el incendio en la cafetera de la sala de descanso?"*
* **Chunks recuperados por FAISS:** 1. *"Protocolo de expulsión de cápsulas incendiarias en el motor hiperespacial."* (Recuperado porque contiene 'incendio').
    2. *"Menú semanal de la sala de descanso: Lunes de café y donas."* (Recuperado porque contiene 'café' y 'sala de descanso').

**El Problema:**
Ninguno de los fragmentos recuperados contiene la solución real para apagar el fuego de la cafetera. Si le pasamos esto directamente al LLM, podría sufrir una alucinación letal y recomendarle al astronauta que "expulse la sala de descanso al hiperespacio".

---

### ¿Cómo detectaríamos y evitaríamos esto en Producción?

Para evitar que el LLM invente respuestas cuando no tiene los datos correctos, implementaríamos las siguientes barreras de seguridad:

**1. Umbral de Similitud (Score Threshold) en FAISS:**
Antes de enviar nada al LLM, evaluamos la puntuación de similitud del coseno de los *chunks* recuperados. Sabemos que las coincidencias perfectas rozan el `1.0`. Si el *Top 1* recuperado tiene un score muy bajo (por ejemplo, `< 0.45`), asumimos que la base de datos no tiene la respuesta. El sistema aborta la llamada al LLM y devuelve un error predeterminado: *"No hay información en el manual sobre este tema"*.

**2. Ingeniería de Prompts (Prompting de Abstención):**
Incluso si los *chunks* superan el umbral por poco, obligamos al LLM a ser consciente de su ignorancia. Añadimos una regla de oro irrompible en el `system_prompt`:
> *"Usa ÚNICAMENTE el contexto proporcionado para responder. Si el contexto no contiene los pasos exactos para responder a la pregunta, DEBES responder textualmente: 'No tengo suficiente información en mis manuales para responder a esto, contacte al Capitán'."*

**3. Sistema de Doble Verificación (Self-Critique Pipeline):**
Añadimos un paso extra invisible para el usuario. Una vez que el LLM genera la respuesta, usamos un segundo prompt más pequeño y barato que evalúa: *"¿La respuesta generada se puede justificar 100%
